# ConvNeXt-Tiny multi-seed runs — Oral cancer histopathology

Single training runs of the same setting differed by 3–5 points, so every setting is trained with **3 different random seeds** and reported as **mean ± SD**.

| Setting | Seeds | Approx. time per run (T4) |
|---|---|---|
| 3-class, NDB-UFES (primary) | 1, 2, 3 | ~10–15 min |
| 4-class | 1, 2, 3 | ~15–25 min |
| 4-class, greyscale | 1, 2, 3 | ~15–25 min |

**Total: ~2.5–3.5 hours** (9 runs). Code: [oral-cancer-detection](https://github.com/Duggineniakhil/oral-cancer-detection) · `scripts/classification/train_classifier.py` and `summarize_seeds.py`.

### Before running
- **Add Input** → your dataset with `splits/` and `preprocessed_normalized/` (the sharpness-matched version).
- **Settings** → Accelerator **GPU T4**, Internet **On**.
- Use **Save Version → Save & Run All** so all 9 runs finish in the background. Runs that already finished in the same session are skipped if you re-run a cell.

## 1. Setup

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU! Turn it on in Settings -> Accelerator before continuing.")

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Duggineniakhil/oral-cancer-detection.git"
CODE_DIR = "/tmp/oral-cancer-detection"

if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)

print(subprocess.run(["git", "-C", CODE_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

matches = list(Path("/kaggle/input").rglob("splits/normalized_dataset.csv"))
assert matches, "Could not find splits/normalized_dataset.csv under /kaggle/input. Did you add the dataset as an input?"

DATA_ROOT = matches[0].parent.parent
OUTPUT_ROOT = Path("/kaggle/working")

os.environ["ORAL_CANCER_DATA_ROOT"] = str(DATA_ROOT)
os.environ["ORAL_CANCER_OUTPUT_ROOT"] = str(OUTPUT_ROOT)

print("Data root:  ", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)

## 2. Train all settings × seeds

In [ ]:
SEEDS = [1, 2, 3]

SETTINGS = {
    "convnext_tiny_3class_224px": "--classes 3",
    "convnext_tiny_4class_224px": "--classes 4",
    "convnext_tiny_4class_gray_224px": "--classes 4 --grayscale",
}

for seed in SEEDS:
    for name, flags in SETTINGS.items():
        run_name = f"{name}_seed{seed}"
        if (OUTPUT_ROOT / "runs" / run_name / "metrics.json").exists():
            print(f"Skipping {run_name} (already done)")
            continue
        print(f"\n################ {run_name} ################")
        get_ipython().system(
            f"cd {CODE_DIR} && python -m scripts.classification.train_classifier "
            f"{flags} --seed {seed} --run-name {run_name}"
        )

## 3. Mean ± SD over seeds

The table is also saved as `runs/multiseed_summary.md`, ready to paste into the README.

In [ ]:
!cd {CODE_DIR} && python -m scripts.classification.summarize_seeds

In [ ]:
import pandas as pd

per_run = pd.read_csv(OUTPUT_ROOT / "runs" / "multiseed_per_run.csv")
per_run.pivot(index="seed", columns="setting", values="balanced_accuracy").mul(100).round(1)

## 4. Download results

`multiseed_results.zip` contains metrics, predictions, training history and plots for all runs. **Model weights are left out**, so it is only a few MB. They remain in this version's Output tab if needed.

In [ ]:
import shutil
from IPython.display import FileLink

src = OUTPUT_ROOT / "runs"
dst = OUTPUT_ROOT / "multiseed_results" / "runs"
shutil.rmtree(dst.parent, ignore_errors=True)
shutil.copytree(src, dst, ignore=shutil.ignore_patterns("*.pt"))

archive = shutil.make_archive(str(OUTPUT_ROOT / "multiseed_results"), "zip", dst.parent, "runs")
print(archive, round(os.path.getsize(archive) / 2**20, 1), "MB")
FileLink("multiseed_results.zip")